## Simple MultiAi Agent Architecture

In [ ]:
import os 
from dotenv import load_dotenv
from typing import TypedDict , Annotated, List , Literal
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,SystemMessage
from langchain_tavily  import TavilySearch
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode,tools_condition


In [3]:
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [4]:
## state defination

class State(TypedDict):
    next_agent:str # which agent should go next 
    

In [5]:
## creating the sample tools
@tool
def search_web(query: str)->str:
    """search the web for information"""
    search = TavilySearch(max_results=3)
    results = search.invoke(query)
    return results

@tool
def write_summary(content:str)->str:
    """write a summary of the content"""
    summary = f"summary of findings:\n\n{content[:500]}..."
    return summary

In [6]:
llm = ChatGroq(model = "qwen/qwen3.8-27b",temperature=0)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001D0124106E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D012411400>, model_name='qwen/qwen3.8-27b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from langgraph.graph import add_messages

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [8]:
## creatinng researcher agent defination

def researcher_agent(state:AgentState):
    """Researcher agent that searches for the information"""
    message = state["messages"]
    #adding system message context
    system_msg = SystemMessage(content = "you are the research assistant. Use the search_web too to find the relevant information.")
    
    ##call llm with tool
    researcher_llm = llm.bind_tools([search_web])
    response = researcher_llm.invoke([system_msg] + message)
    
    ##returing the response to writer
    return {
        "message":[response],
        "next_agent":"writer"
    }

In [9]:
## creating writer agent defination

def writer_agent(state:AgentState):
    """writer agent that writes the summary"""
    
    #adding system message
    sys_msg = SystemMessage(content = "you are a technical writer. Review the conversations and create a clear and concise summary.")
    
    ##adding tool with tool
    
    writing_tool = llm.bind_tools([write_summary])
    response  = writing_tool.invoke([sys_msg] + state["messages"])
    
    ##returing the response to researcher
    return {
        "message":[response],
        "next_agent":"end"
    }
    

In [10]:
#tool executor node 
def execute_tools(state: AgentState):
    """execute any pending tools"""
    message = state['messages']
    last_message = message[-1]
    
    
    if hasattr(last_message,"tool_calls") and last_message.tool_calls:
        tool_node = ToolNode([search_web,write_summary])
        response =  tool_node.invoke(state)
        return response
    return state        